# ============================================================================
# CLASE 1: CONCEPTOS BÁSICOS DE BASES DE DATOS Y CONEXIÓN
# ============================================================================


impotar las paqueterias correspondientes

In [5]:
import sqlite3
from sqlite3 import connect, Error
import json
from datetime import datetime

ahora tenemos que definir algunas cosas antes:


<font color=cyan>**Clase 1: Introducción a SQL y creación de conexiones**

<font color=magenta>**¿Qué es una base de datos?**

Conjunto organizado de datos almacenados y accesibles electrónicamente. Permite guardar, consultar y manipular información de forma estructurada.

<font color=magenta>**¿Qué es SQL?**

Structured Query Language (Lenguaje de Consulta Estructurada). Es el lenguaje estándar para comunicarse con bases de datos relacionales, permitiendo consultar, insertar, actualizar y eliminar datos.

<font color=magenta>**Conexión a bases de datos**

Proceso de establecer comunicación entre una aplicación y un SGBD (Sistema Gestor de Base de Datos) mediante credenciales, host, puerto y nombre de la base de datos.

<font color=magenta>**Creación de tablas básicas**

Instrucción SQL CREATE TABLE que define la estructura de una tabla con columnas, tipos de datos y restricciones básicas (ej. INT, VARCHAR, NOT NULL).


In [9]:
class Clase1_ConceptosBasicos:
    """
    Clase 1: Introducción a SQL y creación de conexiones
    - ¿Qué es una base de datos?
    - ¿Qué es SQL?
    - Conexión a bases de datos
    - Creación de tablas básicas
    """
    
    def __init__(self, nombre_db="estudiantes.db"):
        self.nombre_db = nombre_db
        self.conexion = None
        
    def conectar(self):
        """Crea una conexión a la base de datos SQLite"""
        try:
            self.conexion = sqlite3.connect(self.nombre_db)
            self.cursor = self.conexion.cursor()
            print(f"✓ Conexión exitosa a {self.nombre_db}")
            return True
        except Error as e:
            print(f"✗ Error en conexión: {e}")
            return False
    
    def crear_tabla_estudiantes(self):
        """
        CREATE TABLE - Crea una tabla
        
        Conceptos:
        - CREATE TABLE: comando para crear tabla
        - PRIMARY KEY: identificador único
        - NOT NULL: el valor es obligatorio
        - VARCHAR(n): texto de máximo n caracteres
        - INTEGER: números enteros
        - REAL: números decimales
        """
        sql = """
        CREATE TABLE IF NOT EXISTS estudiantes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre VARCHAR(100) NOT NULL,
            edad INTEGER,
            email VARCHAR(100) NOT NULL,
            grado VARCHAR(50),
            fecha_registro DATE
        );
        """
        try:
            self.cursor.execute(sql)
            self.conexion.commit()
            print("✓ Tabla 'estudiantes' creada")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def ver_estructura_tabla(self):
        """Muestra la estructura de la tabla"""
        sql = "PRAGMA table_info(estudiantes);"
        self.cursor.execute(sql)
        columnas = self.cursor.fetchall()
        
        print("\n📋 ESTRUCTURA DE LA TABLA 'estudiantes':")
        print("-" * 60)
        for col in columnas:
            print(f"  {col[1]:15} | Tipo: {col[2]:10} | Requerido: {'Sí' if col[3] else 'No'}")
    
    def cerrar(self):
        if self.conexion:
            self.conexion.close()
            print("✓ Conexión cerrada")

# ============================================================================
# CLASE 2: INSERCIÓN (INSERT) Y LECTURA (SELECT) BÁSICA
# ============================================================================

<font color=cyan>**Clase 2: Operaciones básicas de datos**

<font color=yellow>**INSERT INTO: insertar registros**

Comando SQL que agrega nuevas filas de datos a una tabla existente. Puede insertar un registro a la vez o múltiples registros en una sola instrucción.

<font color=yellow>**SELECT: consultar datos**

Instrucción fundamental para recuperar información de una o más tablas. Permite especificar qué columnas mostrar y de qué tablas obtener los datos.

<font color=yellow>**WHERE: filtrar datos**

Cláusula utilizada con SELECT, UPDATE o DELETE para establecer condiciones que determinan qué registros serán afectados o mostrados, actuando como filtro.

<font color=yellow>**ORDER BY: ordenar resultados**

Cláusula que organiza los registros resultantes de una consulta en orden ascendente (ASC) o descendente (DESC) basado en una o más columnas especificadas.

In [10]:
class Clase2_InsertarYLeer:
    """
    Clase 2: Operaciones básicas de datos
    - INSERT INTO: insertar registros
    - SELECT: consultar datos
    - WHERE: filtrar datos
    - ORDER BY: ordenar resultados
    """
    
    def __init__(self, nombre_db="estudiantes.db"):
        self.nombre_db = nombre_db
        self.conectar()
    
    def conectar(self):
        self.conexion = sqlite3.connect(self.nombre_db)
        self.cursor = self.conexion.cursor()
    
    def insertar_estudiantes(self, estudiantes):
        """
        INSERT INTO - Inserta datos en la tabla
        
        Sintaxis: INSERT INTO tabla (columnas) VALUES (valores)
        
        Args:
            estudiantes: lista de tuplas con datos del estudiante
        """
        sql = """
        INSERT INTO estudiantes (nombre, edad, email, grado, fecha_registro)
        VALUES (?, ?, ?, ?, ?);
        """
        try:
            self.cursor.executemany(sql, estudiantes)
            self.conexion.commit()
            print(f"✓ {len(estudiantes)} estudiantes insertados")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def seleccionar_todos(self):
        """
        SELECT * - Selecciona todos los registros
        
        * = todas las columnas
        """
        sql = "SELECT * FROM estudiantes;"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def seleccionar_columnas_especificas(self):
        """SELECT específico - Selecciona solo columnas necesarias"""
        sql = "SELECT nombre, email, grado FROM estudiantes;"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def filtrar_por_edad(self, edad_minima):
        """
        WHERE - Filtra registros por condición
        
        Sintaxis: SELECT * FROM tabla WHERE condición
        """
        sql = "SELECT nombre, edad, grado FROM estudiantes WHERE edad >= ?;"
        self.cursor.execute(sql, (edad_minima,))
        return self.cursor.fetchall()
    
    def filtrar_por_grado(self, grado):
        """Filtra estudiantes por grado específico"""
        sql = "SELECT * FROM estudiantes WHERE grado = ?;"
        self.cursor.execute(sql, (grado,))
        return self.cursor.fetchall()
    
    def ordenar_por_edad(self, ascendente=True):
        """
        ORDER BY - Ordena resultados
        
        ASC: ascendente (de menor a mayor)
        DESC: descendente (de mayor a menor)
        """
        orden = "ASC" if ascendente else "DESC"
        sql = f"SELECT nombre, edad FROM estudiantes ORDER BY edad {orden};"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def mostrar_datos(self, datos, titulo):
        """Función auxiliar para mostrar datos formateados"""
        print(f"\n📊 {titulo}")
        print("-" * 60)
        for fila in datos:
            print(f"  {fila}")
    
    def cerrar(self):
        self.conexion.close()




# ============================================================================
# CLASE 3: ACTUALIZACIÓN (UPDATE) Y ELIMINACIÓN (DELETE)
# ============================================================================


<font color=orange>UPDATE: modificar registros existentes</font>

Comando SQL que modifica los valores de uno o varios registros en una tabla. Se utiliza con SET para especificar las columnas a actualizar y WHERE para determinar qué registros se modifican.

<font color=orange>DELETE: eliminar registros</font>

Instrucción SQL que remueve filas completas de una tabla. Es fundamental utilizar la cláusula WHERE para evitar eliminar todos los registros de la tabla.

<font color=orange>LIMIT: limitar cantidad de resultados</font>

Cláusula que restringe el número de filas devueltas por una consulta. Útil para obtener muestras de datos, paginación de resultados o evitar sobrecarga en consultas masivas.



In [18]:

class Clase3_ActualizarYEliminar:
    """
    Clase 3: Modificación y eliminación de datos
    - UPDATE: modificar registros existentes
    - DELETE: eliminar registros
    - LIMIT: limitar cantidad de resultados
    """
    
    def __init__(self, nombre_db="estudiantes.db"):
        self.nombre_db = nombre_db
        self.conectar()
    
    def conectar(self):
        self.conexion = sqlite3.connect(self.nombre_db)
        self.cursor = self.conexion.cursor()
    
    def actualizar_edad_estudiante(self, id_estudiante, nueva_edad):
        """
        UPDATE - Modifica registros existentes
        
        Sintaxis: UPDATE tabla SET columna=valor WHERE condición
        """
        sql = "UPDATE estudiantes SET edad = ? WHERE id = ?;"
        try:
            self.cursor.execute(sql, (nueva_edad, id_estudiante))
            self.conexion.commit()
            print(f"✓ Edad del estudiante ID {id_estudiante} actualizada a {nueva_edad}")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def actualizar_grado_masivo(self, grado_actual, grado_nuevo):
        """Actualiza el grado de múltiples estudiantes"""
        sql = "UPDATE estudiantes SET grado = ? WHERE grado = ?;"
        try:
            self.cursor.execute(sql, (grado_nuevo, grado_actual))
            filas_actualizadas = self.cursor.rowcount
            self.conexion.commit()
            print(f"✓ {filas_actualizadas} registros actualizados")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def eliminar_estudiante_por_id(self, id_estudiante):
        """
        DELETE - Elimina registros
        
        Sintaxis: DELETE FROM tabla WHERE condición
        ⚠️ IMPORTANTE: Siempre usar WHERE para evitar eliminar todo
        """
        sql = "DELETE FROM estudiantes WHERE id = ?;"
        try:
            self.cursor.execute(sql, (id_estudiante,))
            self.conexion.commit()
            print(f"✓ Estudiante ID {id_estudiante} eliminado")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def eliminar_estudiantes_por_edad(self, edad_maxima):
        """Elimina estudiantes menores a una edad específica"""
        sql = "DELETE FROM estudiantes WHERE edad < ?;"
        try:
            self.cursor.execute(sql, (edad_maxima,))
            filas_eliminadas = self.cursor.rowcount
            self.conexion.commit()
            print(f"✓ {filas_eliminadas} estudiantes eliminados")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def obtener_primeros_n_registros(self, n):
        """
        LIMIT - Limita la cantidad de resultados
        
        Sintaxis: SELECT * FROM tabla LIMIT n
        """
        sql = "SELECT * FROM estudiantes LIMIT ?;"
        self.cursor.execute(sql, (n,))
        return self.cursor.fetchall()
    
    def mostrar_datos(self, datos, titulo):
        """Función auxiliar para mostrar datos formateados"""
        print(f"\n📊 {titulo}")
        print("-" * 60)
        for fila in datos:
            print(f"  {fila}")
    
    def cerrar(self):
        self.conexion.close()




# ============================================================================
# CLASE 4: FUNCIONES DE AGREGACIÓN Y AGRUPACIÓN
# ============================================================================

<font color=cyan>**Clase 4: Funciones de agregación y agrupación**</font>

<font color=lightgreen>**COUNT: contar registros**</font>

Función que devuelve el número total de filas que coinciden con una condición. Puede usarse como COUNT(*) para contar todas las filas o COUNT(columna) para contar valores no nulos.

<font color=lightgreen>**SUM: sumar valores**</font>

Función que calcula la suma total de los valores numéricos de una columna. Ignora valores NULL en el cálculo.

<font color=lightgreen>**AVG: promedio**</font>

Función que calcula el valor promedio de una columna numérica. Divide la suma total entre el número de valores no nulos.

<font color=lightgreen>**MIN/MAX: valores mínimos y máximos**</font>

Funciones que encuentran el valor más pequeño (MIN) y el valor más grande (MAX) en una columna. Funcionan con tipos de datos numéricos, texto y fechas.

<font color=lightgreen>**GROUP BY: agrupar datos**</font>

Cláusula que agrupa filas que tienen el mismo valor en una o más columnas, permitiendo aplicar funciones de agregación a cada grupo individualmente.

<font color=lightgreen>**HAVING: filtrar grupos**</font>

Cláusula similar a WHERE pero aplicada a grupos creados con GROUP BY. Permite filtrar los resultados de las funciones de agregación.

In [12]:


class Clase4_FuncionesAgregacion:
    """
    Clase 4: Operaciones de análisis de datos
    - COUNT: contar registros
    - SUM: sumar valores
    - AVG: promedio
    - MIN/MAX: valores mínimos y máximos
    - GROUP BY: agrupar datos
    - HAVING: filtrar grupos
    """
    
    def __init__(self, nombre_db="estudiantes.db"):
        self.nombre_db = nombre_db
        self.conectar()
    
    def conectar(self):
        self.conexion = sqlite3.connect(self.nombre_db)
        self.cursor = self.conexion.cursor()
    
    def contar_estudiantes(self):
        """
        COUNT - Cuenta la cantidad de registros
        
        Sintaxis: SELECT COUNT(*) FROM tabla
        """
        sql = "SELECT COUNT(*) FROM estudiantes;"
        self.cursor.execute(sql)
        resultado = self.cursor.fetchone()
        return resultado[0]
    
    def contar_por_grado(self):
        """
        GROUP BY - Agrupa datos por columna
        
        Sintaxis: SELECT columna, COUNT(*) FROM tabla GROUP BY columna
        """
        sql = "SELECT grado, COUNT(*) as cantidad FROM estudiantes GROUP BY grado;"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def edad_promedio(self):
        """
        AVG - Calcula el promedio
        
        Sintaxis: SELECT AVG(columna) FROM tabla
        """
        sql = "SELECT AVG(edad) FROM estudiantes;"
        self.cursor.execute(sql)
        resultado = self.cursor.fetchone()
        return round(resultado[0], 2) if resultado[0] else 0
    
    def edad_promedio_por_grado(self):
        """AVG con GROUP BY"""
        sql = "SELECT grado, AVG(edad) as edad_promedio FROM estudiantes GROUP BY grado;"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def edad_minima_maxima(self):
        """
        MIN y MAX - Valores extremos
        
        Sintaxis: SELECT MIN(columna), MAX(columna) FROM tabla
        """
        sql = "SELECT MIN(edad), MAX(edad) FROM estudiantes;"
        self.cursor.execute(sql)
        return self.cursor.fetchone()
    
    def suma_edades_por_grado(self):
        """
        SUM - Suma valores
        
        Sintaxis: SELECT SUM(columna) FROM tabla
        """
        sql = "SELECT grado, SUM(edad) as suma_edades FROM estudiantes GROUP BY grado;"
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def estadisticas_completas_por_grado(self):
        """Combina múltiples funciones de agregación"""
        sql = """
        SELECT 
            grado,
            COUNT(*) as cantidad,
            AVG(edad) as edad_promedio,
            MIN(edad) as edad_minima,
            MAX(edad) as edad_maxima
        FROM estudiantes
        GROUP BY grado
        ORDER BY cantidad DESC;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def filtrar_grupos_con_having(self, cantidad_minima):
        """
        HAVING - Filtra grupos (WHERE filtra filas, HAVING filtra grupos)
        
        Sintaxis: SELECT ... GROUP BY ... HAVING condición
        """
        sql = """
        SELECT grado, COUNT(*) as cantidad 
        FROM estudiantes 
        GROUP BY grado 
        HAVING COUNT(*) >= ?
        ORDER BY cantidad DESC;
        """
        self.cursor.execute(sql, (cantidad_minima,))
        return self.cursor.fetchall()
    
    def cerrar(self):
        self.conexion.close()




# ============================================================================
# CLASE 5: UNIONES (JOINS) Y CONSULTAS AVANZADAS
# ============================================================================

<font color=cyan>**Clase 5: Uniones (JOINS) y consultas avanzadas**</font>

<font color=gold>**INNER JOIN: intersección de tablas**</font>

Tipo de JOIN que retorna únicamente los registros que tienen correspondencia en ambas tablas según la condición establecida. Muestra solo la intersección de los datos.

<font color=gold>**LEFT JOIN: todos de la izquierda**</font>

Tipo de JOIN que retorna todos los registros de la tabla izquierda (primera tabla) y los registros coincidentes de la tabla derecha. Cuando no hay coincidencia, muestra NULL en las columnas de la derecha.

<font color=gold>**RIGHT JOIN: todos de la derecha**</font>

Tipo de JOIN que retorna todos los registros de la tabla derecha (segunda tabla) y los registros coincidentes de la tabla izquierda. Es el complemento de LEFT JOIN.

<font color=gold>**Subconsultas**</font>

Consultas anidadas dentro de otra consulta principal. Pueden estar en SELECT, FROM, WHERE o HAVING. Retornan un valor único, una lista de valores o una tabla completa según su ubicación.

<font color=gold>**Alias (AS)**</font>

Palabra reservada que asigna un nombre temporal a una columna, tabla o subconsulta. Mejora la legibilidad de consultas complejas y permite referenciar elementos con nombres más cortos o descriptivos.

In [13]:


class Clase5_UnionesYJoins:
    """
    Clase 5: Consultas avanzadas
    - INNER JOIN: intersección de tablas
    - LEFT JOIN: todos de la izquierda
    - RIGHT JOIN: todos de la derecha
    - Subconsultas
    - Alias (AS)
    """
    
    def __init__(self, nombre_db="estudiantes.db"):
        self.nombre_db = nombre_db
        self.conectar()
    
    def conectar(self):
        self.conexion = sqlite3.connect(self.nombre_db)
        self.cursor = self.conexion.cursor()
    
    def crear_tabla_calificaciones(self):
        """Crea una segunda tabla para ejemplificar JOINS"""
        sql = """
        CREATE TABLE IF NOT EXISTS calificaciones (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            estudiante_id INTEGER NOT NULL,
            materia VARCHAR(100),
            calificacion REAL,
            FOREIGN KEY (estudiante_id) REFERENCES estudiantes(id)
        );
        """
        try:
            self.cursor.execute(sql)
            self.conexion.commit()
            print("✓ Tabla 'calificaciones' creada")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def insertar_calificaciones(self, calificaciones):
        """Inserta datos en la tabla de calificaciones"""
        sql = """
        INSERT INTO calificaciones (estudiante_id, materia, calificacion)
        VALUES (?, ?, ?);
        """
        try:
            self.cursor.executemany(sql, calificaciones)
            self.conexion.commit()
            print(f"✓ {len(calificaciones)} calificaciones insertadas")
        except Error as e:
            print(f"✗ Error: {e}")
    
    def inner_join_ejemplo(self):
        """
        INNER JOIN - Combina dos tablas por una condición común
        Retorna solo registros que coinciden en ambas tablas
        
        Sintaxis: SELECT ... FROM tabla1 INNER JOIN tabla2 ON tabla1.id = tabla2.id
        """
        sql = """
        SELECT 
            e.nombre,
            e.grado,
            c.materia,
            c.calificacion
        FROM estudiantes e
        INNER JOIN calificaciones c ON e.id = c.estudiante_id
        ORDER BY e.nombre;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def left_join_ejemplo(self):
        """
        LEFT JOIN - Retorna todos los registros de la izquierda
        y los coincidentes de la derecha (NULL si no hay coincidencia)
        """
        sql = """
        SELECT 
            e.nombre,
            c.materia,
            c.calificacion
        FROM estudiantes e
        LEFT JOIN calificaciones c ON e.id = c.estudiante_id
        ORDER BY e.nombre;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def usar_alias(self):
        """
        Alias (AS) - Renombra columnas o tablas temporalmente
        
        Utilidad:
        - Acortar nombres largos
        - Hacer consultas más legibles
        - Distinguir columnas con mismo nombre de diferentes tablas
        """
        sql = """
        SELECT 
            e.nombre AS nombre_estudiante,
            COUNT(c.id) AS cantidad_calificaciones,
            ROUND(AVG(c.calificacion), 2) AS promedio
        FROM estudiantes e
        LEFT JOIN calificaciones c ON e.id = c.estudiante_id
        GROUP BY e.id, e.nombre;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def subconsulta_ejemplo(self):
        """
        Subconsulta - Una consulta dentro de otra consulta
        
        Sintaxis: SELECT * FROM tabla WHERE columna IN (SELECT ...)
        """
        sql = """
        SELECT 
            nombre,
            grado,
            edad
        FROM estudiantes
        WHERE edad > (SELECT AVG(edad) FROM estudiantes)
        ORDER BY edad DESC;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def consulta_compleja(self):
        """Combina JOINS, GROUP BY, HAVING y ORDER BY"""
        sql = """
        SELECT 
            e.nombre,
            e.grado,
            COUNT(c.id) AS cantidad_calificaciones,
            ROUND(AVG(c.calificacion), 2) AS promedio_calificacion
        FROM estudiantes e
        LEFT JOIN calificaciones c ON e.id = c.estudiante_id
        GROUP BY e.id, e.nombre, e.grado
        HAVING COUNT(c.id) > 0
        ORDER BY promedio_calificacion DESC;
        """
        self.cursor.execute(sql)
        return self.cursor.fetchall()
    
    def cerrar(self):
        self.conexion.close()




# ============================================================================
# FUNCIÓN PRINCIPAL - DEMOSTRACIÓN DE TODAS LAS CLASES
# ============================================================================

In [15]:
def main():
    """Ejecuta la guía completa de SQL"""
    
    print("=" * 70)
    print("GUÍA DE ESTUDIO DE SQL DESDE CERO - 5 CLASES")
    print("=" * 70)
    
    # CLASE 1: Conceptos básicos
    print("\n🔷 CLASE 1: CONCEPTOS BÁSICOS Y CONEXIÓN")
    print("-" * 70)
    clase1 = Clase1_ConceptosBasicos("sql_guia.db")
    clase1.conectar()
    clase1.crear_tabla_estudiantes()
    clase1.ver_estructura_tabla()
    clase1.cerrar()
    
    # CLASE 2: Insert y Select
    print("\n\n🔷 CLASE 2: INSERTAR Y LEER DATOS")
    print("-" * 70)
    clase2 = Clase2_InsertarYLeer("sql_guia.db")
    
    # Datos de ejemplo
    estudiantes = [
        ("Juan García", 18, "juan@email.com", "1° Año", "2024-01-15"),
        ("María López", 17, "maria@email.com", "1° Año", "2024-01-16"),
        ("Carlos Rodríguez", 19, "carlos@email.com", "2° Año", "2024-01-17"),
        ("Ana Martínez", 18, "ana@email.com", "2° Año", "2024-01-18"),
        ("Pedro Sánchez", 20, "pedro@email.com", "3° Año", "2024-01-19"),
    ]
    
    clase2.insertar_estudiantes(estudiantes)
    
    datos = clase2.seleccionar_todos()
    clase2.mostrar_datos(datos, "SELECT * FROM estudiantes")
    
    datos = clase2.filtrar_por_edad(18)
    clase2.mostrar_datos(datos, "Estudiantes con edad >= 18")
    
    datos = clase2.ordenar_por_edad(ascendente=True)
    clase2.mostrar_datos(datos, "Estudiantes ordenados por edad (ascendente)")
    
    clase2.cerrar()
    
    # CLASE 3: Update y Delete
    print("\n\n🔷 CLASE 3: ACTUALIZAR Y ELIMINAR DATOS")
    print("-" * 70)
    clase3 = Clase3_ActualizarYEliminar("sql_guia.db")
    
    clase3.actualizar_edad_estudiante(1, 19)
    clase3.actualizar_grado_masivo("1° Año", "1° Año Actualizado")
    
    primeros = clase3.obtener_primeros_n_registros(3)
    clase3.mostrar_datos(primeros, "Primeros 3 registros (LIMIT 3)")
    
    clase3.cerrar()
    
    # CLASE 4: Funciones de agregación
    print("\n\n🔷 CLASE 4: FUNCIONES DE AGREGACIÓN")
    print("-" * 70)
    clase4 = Clase4_FuncionesAgregacion("sql_guia.db")
    
    total = clase4.contar_estudiantes()
    print(f"\n📈 Total de estudiantes: {total}")
    
    edad_prom = clase4.edad_promedio()
    print(f"📊 Edad promedio: {edad_prom}")
    
    min_edad, max_edad = clase4.edad_minima_maxima()
    print(f"📊 Edad mínima: {min_edad}, Edad máxima: {max_edad}")
    
    print("\n📋 Cantidad de estudiantes por grado:")
    datos = clase4.contar_por_grado()
    for grado, cantidad in datos:
        print(f"   {grado}: {cantidad}")
    
    print("\n📋 Estadísticas completas por grado:")
    datos = clase4.estadisticas_completas_por_grado()
    print("-" * 70)
    for fila in datos:
        print(f"  Grado: {fila[0]:<15} | Cantidad: {fila[1]:<5} | Edad Prom: {fila[2]:<6} | Min: {fila[3]:<5} | Max: {fila[4]}")
    
    clase4.cerrar()
    
    # CLASE 5: Joins
    print("\n\n🔷 CLASE 5: UNIONES (JOINS) Y CONSULTAS AVANZADAS")
    print("-" * 70)
    clase5 = Clase5_UnionesYJoins("sql_guia.db")
    
    clase5.crear_tabla_calificaciones()
    
    # Datos de calificaciones
    calificaciones = [
        (1, "Matemáticas", 8.5),
        (1, "Historia", 7.0),
        (2, "Matemáticas", 9.0),
        (3, "Inglés", 8.0),
        (4, "Matemáticas", 7.5),
        (4, "Historia", 8.5),
        (5, "Inglés", 9.5),
    ]
    
    clase5.insertar_calificaciones(calificaciones)
    
    print("\n📋 INNER JOIN - Estudiantes con calificaciones:")
    datos = clase5.inner_join_ejemplo()
    for fila in datos:
        print(f"   {fila[0]:<20} | Grado: {fila[1]:<15} | {fila[2]:<15} | Calificación: {fila[3]}")
    
    print("\n📋 Uso de ALIAS - Información resumida:")
    datos = clase5.usar_alias()
    for fila in datos:
        print(f"   {fila[0]:<20} | Calificaciones: {fila[1]:<5} | Promedio: {fila[2]}")
    
    print("\n📋 Subconsulta - Estudiantes con edad superior al promedio:")
    datos = clase5.subconsulta_ejemplo()
    for fila in datos:
        print(f"   {fila[0]:<20} | Grado: {fila[1]:<15} | Edad: {fila[2]}")
    
    print("\n📋 Consulta compleja - Top de estudiantes por promedio:")
    datos = clase5.consulta_compleja()
    for fila in datos:
        print(f"   {fila[0]:<20} | Grado: {fila[1]:<15} | Promedio: {fila[3]}")
    
    clase5.cerrar()
    
    print("\n" + "=" * 70)
    print("✓ GUÍA COMPLETADA - ¡Felicidades por aprender SQL! 🎉")
    print("=" * 70)




In [19]:
if __name__ == "__main__":
    main()

GUÍA DE ESTUDIO DE SQL DESDE CERO - 5 CLASES

🔷 CLASE 1: CONCEPTOS BÁSICOS Y CONEXIÓN
----------------------------------------------------------------------
✓ Conexión exitosa a sql_guia.db
✓ Tabla 'estudiantes' creada

📋 ESTRUCTURA DE LA TABLA 'estudiantes':
------------------------------------------------------------
  id              | Tipo: INTEGER    | Requerido: No
  nombre          | Tipo: VARCHAR(100) | Requerido: Sí
  edad            | Tipo: INTEGER    | Requerido: No
  email           | Tipo: VARCHAR(100) | Requerido: Sí
  grado           | Tipo: VARCHAR(50) | Requerido: No
  fecha_registro  | Tipo: DATE       | Requerido: No
✓ Conexión cerrada


🔷 CLASE 2: INSERTAR Y LEER DATOS
----------------------------------------------------------------------
✓ 5 estudiantes insertados

📊 SELECT * FROM estudiantes
------------------------------------------------------------
  (1, 'Juan García', 19, 'juan@email.com', '1° Año Actualizado', '2024-01-15')
  (2, 'María López', 17, 'maria@ema